# 05 — Redes neuronales con PyTorch

**Módulo 2 · Algoritmos avanzados de ML (+ MLflow)**

Este es el notebook **más importante** del módulo. Construimos la intuición de una
red neuronal desde la **neurona simple** hasta un **perceptrón multicapa (MLP)**,
con **diagramas dibujados en código** (sin imágenes externas) y un balance entre
concepto y matemática. Recorremos:

1. La neurona simple → perceptrón → MLP.
2. Funciones de activación y sus derivadas.
3. El problema del **desvanecimiento del gradiente**.
4. **Decisiones de arquitectura ANTES de programar** (checklist).
5. **Forward pass y backpropagation** (regla de la cadena).
6. El algoritmo **SGD explicado con imágenes**.
7. **Regularización** (dropout, weight decay, early stopping, batchnorm).
8. **Implementación en PyTorch — dos casos**: regresión y clasificación,
   registrados en **MLflow**.

> La idea lleva: **el concepto primero, la fórmula compacta después, y los
> diagramas para iluminar**. Sin muros de ecuaciones.


In [ ]:

import os, sys, warnings
warnings.filterwarnings("ignore")

# Hacemos importable utils/ tanto si el notebook corre desde notebooks/ como
# desde la raíz del repositorio.
_here = os.getcwd()
for cand in (os.path.join(_here, "..", "utils"), os.path.join(_here, "utils"),
             os.path.join(_here, "..", "..", "module2-advanced-ml", "utils")):
    cand = os.path.abspath(cand)
    if os.path.isdir(cand) and cand not in sys.path:
        sys.path.insert(0, cand)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from mlflow_helpers import setup_mlflow, log_and_register, register_best_run, registry_available

np.random.seed(42)
print("Versión de MLflow:", mlflow.__version__)


In [ ]:

import torch
import torch.nn as nn
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
torch.manual_seed(42)


## 1. La neurona simple

**Intuición:** una neurona artificial recibe varias entradas, las pondera, las
suma, le añade un *sesgo* (bias) y pasa el resultado por una **función de
activación** que decide cuánto "se enciende". Apilando muchas neuronas en capas
obtenemos una red capaz de aproximar funciones muy complejas.

Compactamente, para entradas $\mathbf{x}=(x_1,\dots,x_n)$, pesos $\mathbf{w}$ y
sesgo $b$:

$$
z = \mathbf{w}^\top \mathbf{x} + b, \qquad a = \phi(z).
$$

$z$ es la **pre-activación** (una combinación lineal) y $a$ la **activación** (la
salida tras la no linealidad $\phi$). Dibujemos esa neurona.


In [ ]:

# Diagrama de una neurona simple: x1..xn -> pesos -> suma -> +bias -> activación -> salida
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axis("off")

inputs = ["$x_1$", "$x_2$", "$x_3$", r"$\vdots$", "$x_n$"]
y_in = np.linspace(4.5, 0.5, len(inputs))
sum_xy = (5.2, 2.5)     # nodo de la suma
act_xy = (7.2, 2.5)     # nodo de activación
out_xy = (9.2, 2.5)     # salida

# Nodos de entrada y flechas ponderadas hacia la suma
for label, y in zip(inputs, y_in):
    ax.scatter(1.0, y, s=900, c="#cfe8ff", edgecolors="#1f77b4", zorder=3)
    ax.text(1.0, y, label, ha="center", va="center", fontsize=12)
    if label != r"$\vdots$":
        ax.annotate("", xy=sum_xy, xytext=(1.4, y),
                    arrowprops=dict(arrowstyle="->", color="gray"))
        ax.text((1.4 + sum_xy[0]) / 2, (y + sum_xy[1]) / 2 + 0.12,
                r"$w$", color="#d62728", fontsize=10)

# Nodo suma (Σ) con el bias entrando
ax.scatter(*sum_xy, s=1500, c="#fff2cc", edgecolors="#b8860b", zorder=3)
ax.text(*sum_xy, r"$\sum$", ha="center", va="center", fontsize=16)
ax.scatter(sum_xy[0], sum_xy[1] + 2.0, s=700, c="#e2f0d9", edgecolors="green", zorder=3)
ax.text(sum_xy[0], sum_xy[1] + 2.0, "$b$", ha="center", va="center", fontsize=12)
ax.annotate("", xy=sum_xy, xytext=(sum_xy[0], sum_xy[1] + 1.6),
            arrowprops=dict(arrowstyle="->", color="green"))

# Suma -> activación
ax.annotate("", xy=act_xy, xytext=(sum_xy[0] + 0.4, sum_xy[1]),
            arrowprops=dict(arrowstyle="->", color="black"))
ax.text((sum_xy[0] + act_xy[0]) / 2, sum_xy[1] + 0.25, "$z$", fontsize=12)
ax.scatter(*act_xy, s=1500, c="#f4cccc", edgecolors="#cc0000", zorder=3)
ax.text(*act_xy, r"$\phi$", ha="center", va="center", fontsize=15)

# Activación -> salida
ax.annotate("", xy=out_xy, xytext=(act_xy[0] + 0.4, act_xy[1]),
            arrowprops=dict(arrowstyle="->", color="black"))
ax.text((act_xy[0] + out_xy[0]) / 2, act_xy[1] + 0.25, "$a$", fontsize=12)
ax.text(out_xy[0], out_xy[1], "salida", ha="center", va="center", fontsize=11,
        bbox=dict(boxstyle="round", fc="#ddd"))

ax.text(5.2, 5.4, r"$z = \mathbf{w}^\top \mathbf{x} + b \quad\to\quad a = \phi(z)$",
        ha="center", fontsize=14)
ax.set_xlim(0, 10.2); ax.set_ylim(-0.3, 6)
plt.title("La neurona simple", fontsize=13)
plt.tight_layout(); plt.show()


### Del perceptrón al MLP

**Intuición:** una sola neurona (el *perceptrón*) sólo traza una **frontera
lineal**. Apilando capas de neuronas con activaciones no lineales, cada capa
transforma el espacio y la siguiente combina esas transformaciones, hasta que la
red puede separar datos que ninguna recta podría. A esto lo llamamos **perceptrón
multicapa (MLP)**: una capa de entrada, una o más **capas ocultas** y una capa de
salida.


In [ ]:

# Diagrama de una pequeña red por capas (MLP): 3 entradas -> 4 ocultas -> 4 ocultas -> 1 salida
def draw_layer(ax, x, n, color, prefix):
    ys = np.linspace(0.5, 4.5, n)
    coords = []
    for i, y in enumerate(ys):
        ax.scatter(x, y, s=600, c=color, edgecolors="k", zorder=3)
        coords.append((x, y))
    return coords

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.axis("off")
layers = [draw_layer(ax, 1, 3, "#cfe8ff", "in"),
          draw_layer(ax, 4, 4, "#fff2cc", "h1"),
          draw_layer(ax, 7, 4, "#fff2cc", "h2"),
          draw_layer(ax, 10, 1, "#f4cccc", "out")]
# Conexiones totalmente conectadas entre capas consecutivas
for a_layer, b_layer in zip(layers[:-1], layers[1:]):
    for (xa, ya) in a_layer:
        for (xb, yb) in b_layer:
            ax.plot([xa, xb], [ya, yb], color="gray", lw=0.4, zorder=1)

names = ["entrada\n(3)", "oculta 1\n(4)", "oculta 2\n(4)", "salida\n(1)"]
for x, name in zip([1, 4, 7, 10], names):
    ax.text(x, 5.1, name, ha="center", fontsize=10)
ax.set_xlim(0, 11); ax.set_ylim(0, 5.6)
plt.title("Perceptrón multicapa (MLP): capas totalmente conectadas", fontsize=13)
plt.tight_layout(); plt.show()


## 2. Funciones de activación

**Intuición:** sin no linealidad, apilar capas equivale a una sola capa lineal.
La activación es lo que da poder expresivo a la red. Estas son las más usadas:

| Nombre | Fórmula | Rango | Nota |
|---|---|---|---|
| Sigmoid | $\sigma(z)=\frac{1}{1+e^{-z}}$ | $(0,1)$ | satura → gradientes que se desvanecen |
| Tanh | $\tanh(z)$ | $(-1,1)$ | centrada en cero |
| ReLU | $\max(0,z)$ | $[0,\infty)$ | barata, esparsa; neuronas "muertas" |
| Leaky ReLU | $\max(\alpha z, z)$ | $\mathbb{R}$ | evita neuronas muertas |
| GELU | $z\,\Phi(z)$ | $\mathbb{R}$ | ReLU suave; usada en Transformers |

Dibujemos cada activación **y su derivada** (clave para entender el gradiente).


In [ ]:

import torch.nn.functional as F

z = torch.linspace(-6, 6, 400, requires_grad=True)

def deriv(fn, z):
    z = z.clone().detach().requires_grad_(True)
    y = fn(z)
    y.sum().backward()
    return y.detach(), z.grad.detach()

acts = {
    "sigmoid":    lambda t: torch.sigmoid(t),
    "tanh":       lambda t: torch.tanh(t),
    "relu":       lambda t: F.relu(t),
    "leaky_relu": lambda t: F.leaky_relu(t, negative_slope=0.1),
    "gelu":       lambda t: F.gelu(t),
}

fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharex=True)
zz = z.detach().numpy()
for j, (name, fn) in enumerate(acts.items()):
    y, g = deriv(fn, z)
    axes[0, j].plot(zz, y.numpy(), color="#1f77b4")
    axes[0, j].set_title(name); axes[0, j].axhline(0, color="k", lw=0.4)
    axes[0, j].axvline(0, color="k", lw=0.4)
    axes[1, j].plot(zz, g.numpy(), color="#d62728")
    axes[1, j].set_title(f"{name}'  (derivada)"); axes[1, j].axhline(0, color="k", lw=0.4)
    axes[1, j].axvline(0, color="k", lw=0.4)
axes[0, 0].set_ylabel("activación  $\\phi(z)$")
axes[1, 0].set_ylabel("derivada  $\\phi'(z)$")
plt.suptitle("Funciones de activación (arriba) y sus derivadas (abajo)", fontsize=13)
plt.tight_layout(); plt.show()


**Pros y contras (resumen práctico):**

| Activación | Pros | Contras |
|---|---|---|
| Sigmoid | salida en $(0,1)$, útil como probabilidad final | satura, derivada máx. $0.25$ → desvanecimiento |
| Tanh | centrada en cero (mejor que sigmoid en capas ocultas) | sigue saturando en los extremos |
| ReLU | rápida, no satura para $z>0$, induce esparsidad | "neuronas muertas" si $z<0$ siempre |
| Leaky ReLU | arregla las neuronas muertas (pendiente pequeña en $z<0$) | un hiperparámetro extra ($\alpha$) |
| GELU | suave, muy buen rendimiento en redes profundas | algo más cara de computar |

**Regla de oro:** usa **ReLU** (o GELU en redes grandes) en las capas ocultas, y
reserva **sigmoid/softmax** para la **salida** de clasificación.


## 3. El problema del desvanecimiento del gradiente

**Intuición:** para entrenar, el gradiente del error viaja *hacia atrás* capa por
capa, **multiplicándose** por la derivada de la activación en cada paso. Si esas
derivadas son pequeñas (como en sigmoid/tanh, que saturan), el producto se hace
**diminuto** y las primeras capas casi no aprenden. Eso es el *desvanecimiento del
gradiente*.

El dato clave: la derivada de la sigmoid nunca supera **0.25**. Encadenando $L$
capas, el gradiente se escala aproximadamente por $(0.25)^L$ — que cae a cero muy
rápido. Veámoslo.


In [ ]:

# (a) Saturación: la derivada de sigmoid/tanh tiende a 0 en los extremos.
zz = torch.linspace(-8, 8, 400)
sig = torch.sigmoid(zz); dsig = sig * (1 - sig)
th = torch.tanh(zz); dth = 1 - th**2

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(zz, dsig, label="sigmoid'  (máx 0.25)", color="#1f77b4")
axes[0].plot(zz, dth, label="tanh'  (máx 1.0)", color="#ff7f0e")
axes[0].axhline(0.25, color="#1f77b4", ls="--", lw=0.8)
axes[0].set_title("Derivadas que saturan → casi 0 en los extremos")
axes[0].set_xlabel("z"); axes[0].set_ylabel("derivada"); axes[0].legend()

# (b) Cómo encoge la magnitud del gradiente al atravesar muchas capas.
layers = np.arange(1, 21)
for dmax, name, c in [(0.25, "sigmoid (deriv≈0.25)", "#1f77b4"),
                      (1.0, "tanh (deriv≈1.0, mejor caso)", "#ff7f0e"),
                      (1.0, "ReLU (deriv=1 para z>0)", "#2ca02c")]:
    axes[1].plot(layers, dmax ** layers, marker="o", ms=3, label=name, color=c)
axes[1].set_yscale("log")
axes[1].set_title("Magnitud del gradiente ≈ (derivada)$^{n}$,  n = # de capas")
axes[1].set_xlabel("# de capas atravesadas"); axes[1].set_ylabel("factor del gradiente (log)")
axes[1].legend()
plt.tight_layout(); plt.show()


**Remedios habituales:**

- **ReLU (y variantes):** su derivada es $1$ para $z>0$, así que no encoge el
  gradiente — el remedio más simple y efectivo.
- **BatchNorm:** normaliza las pre-activaciones por mini-batch, manteniéndolas en
  una zona donde las derivadas no saturan.
- **Conexiones residuales (skip connections):** crean un "atajo" $x + f(x)$ por el
  que el gradiente fluye directo, base de las redes muy profundas (ResNet).
- **Inicialización cuidada** (Xavier/He): fija la escala inicial de los pesos para
  que las señales no se atenúen ni exploten al propagarse.


## 4. Decisiones de arquitectura ANTES de programar

Antes de escribir una sola línea de PyTorch conviene decidir la arquitectura. Esta
es una **checklist práctica**:

| Decisión | Pregunta guía | Default razonable |
|---|---|---|
| **# de capas** (profundidad) | ¿problema simple o complejo? | empieza con 1–3 ocultas |
| **# de neuronas** (ancho) | ¿cuánta capacidad necesito? | 32–256 por capa; baja hacia la salida |
| **Activación oculta** | ¿qué evita el desvanecimiento? | **ReLU** (o GELU) |
| **Capa de SALIDA** | ¿qué predigo? | **regresión → lineal** (sin activación); **clasif. binaria → 1 neurona + sigmoid**; **multiclase → K neuronas + softmax** |
| **Función de PÉRDIDA** | acorde a la salida | **regresión → MSE**; **binaria → BCE**; **multiclase → cross-entropy** |
| **Optimizador** | ¿robusto por defecto? | **Adam** (o SGD+momentum si quieres exprimir) |
| **Learning rate** | el hiperparámetro más sensible | $10^{-3}$ con Adam; ajústalo primero |
| **Batch size** | ¿memoria vs estabilidad? | 32–256 |
| **Épocas** | ¿cuándo parar? | muchas + **early stopping** |
| **Regularización** | ¿señales de sobreajuste? | dropout, weight decay, batchnorm |

> **Par output ↔ pérdida (lo más importante de recordar):**
> - **Regresión:** salida **lineal** + **MSE**.
> - **Clasificación binaria:** salida con **1 logit** + **BCEWithLogits**.
> - **Clasificación multiclase:** **K logits** + **CrossEntropy**.


## 5. Forward pass y backpropagation

**Forward pass (intuición):** los datos entran y avanzan capa por capa. Para la
capa $\ell$:

$$
z^{(\ell)} = W^{(\ell)} a^{(\ell-1)} + b^{(\ell)},\qquad
a^{(\ell)} = \phi\big(z^{(\ell)}\big),\qquad a^{(0)} = x.
$$

**Backpropagation (intuición):** una vez calculada la pérdida, queremos saber
*cuánto contribuye cada peso al error*. La **regla de la cadena** nos deja
calcular ese gradiente eficientemente propagándolo **hacia atrás**. Definiendo el
error por capa $\delta^{(\ell)} = \partial L / \partial z^{(\ell)}$:

$$
\delta^{(L)} = \nabla_a L \odot \phi'(z^{(L)}),\qquad
\delta^{(\ell)} = \big( W^{(\ell+1)\top} \delta^{(\ell+1)} \big) \odot \phi'(z^{(\ell)}),
$$

y los gradientes que actualizan los parámetros:

$$
\frac{\partial L}{\partial W^{(\ell)}} = \delta^{(\ell)} a^{(\ell-1)\top},\qquad
\frac{\partial L}{\partial b^{(\ell)}} = \delta^{(\ell)} .
$$

El **autograd** de PyTorch construye el grafo de cómputo y hace todo esto
automáticamente al llamar `loss.backward()`.


In [ ]:

# Diagrama del flujo de gradiente: forward (→) y backward (←) por una red pequeña.
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.axis("off")
boxes = ["$x$", "capa 1\n$z^{(1)}, a^{(1)}$", "capa 2\n$z^{(2)}, a^{(2)}$",
         "salida\n$\\hat y$", "pérdida\n$L$"]
xs = np.linspace(0.5, 9.5, len(boxes))
for x, b in zip(xs, boxes):
    ax.text(x, 1.5, b, ha="center", va="center", fontsize=10,
            bbox=dict(boxstyle="round", fc="#cfe8ff", ec="#1f77b4"))
# forward (arriba, azul)
for xa, xb in zip(xs[:-1], xs[1:]):
    ax.annotate("", xy=(xb - 0.6, 1.85), xytext=(xa + 0.6, 1.85),
                arrowprops=dict(arrowstyle="->", color="#1f77b4", lw=1.6))
ax.text(xs.mean(), 2.5, "forward  →  (predicción)", ha="center", color="#1f77b4", fontsize=11)
# backward (abajo, rojo)
for xa, xb in zip(xs[:-1], xs[1:]):
    ax.annotate("", xy=(xa + 0.6, 1.15), xytext=(xb - 0.6, 1.15),
                arrowprops=dict(arrowstyle="->", color="#d62728", lw=1.6))
ax.text(xs.mean(), 0.45, r"backward  ←  (gradientes vía regla de la cadena)",
        ha="center", color="#d62728", fontsize=11)
ax.set_xlim(0, 10); ax.set_ylim(0, 3)
plt.title("Forward pass y backpropagation", fontsize=13)
plt.tight_layout(); plt.show()


## 6. El algoritmo SGD explicado con imágenes

**Intuición:** entrenar es *bajar una montaña* (la superficie de pérdida) dando
pasos en la dirección de máxima pendiente descendente, que es el **gradiente
negativo**. La regla de actualización del descenso de gradiente es:

$$
\theta \leftarrow \theta - \eta\,\nabla_\theta L,
$$

donde $\eta$ es la **tasa de aprendizaje** (el tamaño del paso). El "estocástico"
(SGD) viene de estimar el gradiente con un **mini-batch** en vez de todos los
datos. Veamos tres imágenes.


In [ ]:

# (a) Superficie de pérdida 2D con el camino del descenso de gradiente (quiver).
def loss(w):  # cuenco elíptico simple
    return 0.5 * (w[0]**2 / 3.0 + w[1]**2)
def grad(w):
    return np.array([w[0] / 3.0, w[1] * 2.0])

w = np.array([5.0, 4.0]); eta = 0.25; path = [w.copy()]
for _ in range(18):
    w = w - eta * grad(w); path.append(w.copy())
path = np.array(path)

gx, gy = np.meshgrid(np.linspace(-6, 6, 200), np.linspace(-5, 5, 200))
gz = 0.5 * (gx**2 / 3.0 + gy**2)
plt.figure(figsize=(7, 5.5))
cs = plt.contour(gx, gy, gz, levels=20, cmap="viridis")
plt.clabel(cs, inline=True, fontsize=7)
plt.quiver(path[:-1, 0], path[:-1, 1],
           path[1:, 0] - path[:-1, 0], path[1:, 1] - path[:-1, 1],
           angles="xy", scale_units="xy", scale=1, color="red", width=0.005)
plt.scatter(*path[0], c="red", s=60, label="inicio")
plt.scatter(0, 0, c="black", marker="*", s=160, label="mínimo")
plt.title("Descenso de gradiente sobre la superficie de pérdida")
plt.xlabel(r"$\theta_1$"); plt.ylabel(r"$\theta_2$"); plt.legend()
plt.tight_layout(); plt.show()


In [ ]:

# (b) Efecto de la learning rate sobre una parábola 1D: muy pequeña / buena / muy grande.
def f(x): return x**2
def df(x): return 2*x
xs = np.linspace(-5, 5, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, (eta, title) in zip(axes, [(0.05, "muy pequeña (lenta)"),
                                   (0.4, "buena (converge)"),
                                   (1.02, "muy grande (diverge)")]):
    ax.plot(xs, f(xs), color="#999")
    x = 4.5
    for _ in range(12):
        ax.scatter(x, f(x), color="red", s=25, zorder=3)
        x_new = x - eta * df(x)
        ax.annotate("", xy=(x_new, f(x_new)), xytext=(x, f(x)),
                    arrowprops=dict(arrowstyle="->", color="red", lw=0.8))
        x = x_new
        if abs(x) > 6:  # se escapó
            break
    ax.set_title(f"η = {eta}\n{title}"); ax.set_xlabel("θ"); ax.set_ylabel("L(θ)")
plt.suptitle("Efecto de la tasa de aprendizaje", fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:

# (c) Batch vs mini-batch vs estocástico: cómo de "ruidoso" es el camino al mínimo.
np.random.seed(0)
def noisy_path(noise):
    w = np.array([5.0, 4.0]); pts = [w.copy()]
    for _ in range(40):
        g = grad(w) + np.random.randn(2) * noise
        w = w - 0.18 * g; pts.append(w.copy())
    return np.array(pts)

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.contour(gx, gy, gz, levels=18, cmap="Greys", linewidths=0.5)
for noise, name, c in [(0.0, "batch completo (suave)", "#1f77b4"),
                       (0.6, "mini-batch (algo de ruido)", "#ff7f0e"),
                       (1.8, "estocástico (muy ruidoso)", "#2ca02c")]:
    p = noisy_path(noise)
    ax.plot(p[:, 0], p[:, 1], marker="o", ms=2, label=name, color=c, alpha=0.8)
ax.scatter(0, 0, c="black", marker="*", s=160)
ax.set_title("Batch vs mini-batch vs estocástico")
ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$"); ax.legend()
plt.tight_layout(); plt.show()


**Más allá del SGD básico:** dos mejoras muy usadas.

- **Momentum:** acumula una *velocidad* para amortiguar oscilaciones y acelerar en
  direcciones consistentes:
$$
v_t = \mu v_{t-1} + g_t,\qquad \theta_{t+1} = \theta_t - \eta\, v_t .
$$
- **Adam:** tasa de aprendizaje *adaptativa por parámetro*, combinando estimaciones
  del 1er y 2º momento del gradiente. Es un default robusto:
$$
m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t,\quad
v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2,\quad
\theta_{t+1} = \theta_t - \eta\,\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.
$$


## 7. Regularización en redes neuronales

Las redes tienen muchos parámetros y sobreajustan con facilidad. Cuatro técnicas
clave y **cómo aparecen en PyTorch**:

- **Dropout:** apaga al azar una fracción $p$ de activaciones en entrenamiento;
  evita la co-adaptación y actúa como un ensemble.
  `nn.Dropout(p=0.3)`.
- **Weight decay (L2):** añade $\frac{\lambda}{2}\|\theta\|^2$ a la pérdida,
  encogiendo los pesos.
  `torch.optim.Adam(..., weight_decay=1e-4)`.
- **Early stopping:** detiene el entrenamiento cuando la pérdida de validación deja
  de mejorar. Se implementa con un contador de "paciencia" en el bucle.
- **BatchNorm:** normaliza las pre-activaciones por mini-batch, estabilizando y
  acelerando el entrenamiento (y mitigando el desvanecimiento).
  `nn.BatchNorm1d(num_features)`.

A continuación las usamos todas en los dos casos prácticos.


## 8. Implementación en PyTorch — dos casos

Construimos un **bucle de entrenamiento genérico** (con validación y early
stopping) y lo reutilizamos en:

- **Caso A — Regresión:** California housing, MLP con **salida lineal + MSE**.
- **Caso B — Clasificación:** breast cancer, MLP con **sigmoid + BCE**.

Ambos registran params/métricas/modelo en **MLflow** y los registran en el
registry vía los helpers existentes.


In [ ]:

from torch.utils.data import TensorDataset, DataLoader

def make_loaders(X_tr, y_tr, X_val, y_val, X_te, y_te, batch=32, target_2d=True):
    # Crea DataLoaders de train/val/test a partir de arrays numpy.
    def to_ds(Xa, ya):
        yt = torch.tensor(ya, dtype=torch.float32)
        if target_2d:
            yt = yt.unsqueeze(1)
        return TensorDataset(torch.tensor(Xa, dtype=torch.float32), yt)
    return (DataLoader(to_ds(X_tr, y_tr), batch_size=batch, shuffle=True),
            DataLoader(to_ds(X_val, y_val), batch_size=256, shuffle=False),
            DataLoader(to_ds(X_te, y_te), batch_size=256, shuffle=False))


class MLP(nn.Module):
    # MLP genérico con BatchNorm + ReLU + Dropout. La capa de salida (out_dim,
    # sin activación) la decide la tarea: 1 logit para regresión o clasif. binaria.
    def __init__(self, in_dim, hidden=(64, 32), out_dim=1, p_drop=0.3):
        super().__init__()
        layers, d = [], in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(p_drop)]
            d = h
        layers += [nn.Linear(d, out_dim)]   # salida lineal (logits / valor)
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


In [ ]:

def train_loop(model, train_dl, val_dl, criterion, optimizer, epochs, patience,
               eval_metric):
    # Bucle de entrenamiento genérico con validación y early stopping.
    # eval_metric(model, dl) -> dict con métricas extra por época (ej. RMSE, accuracy).
    # Devuelve (history, best_state). history["val_metric"] es una lista de dicts
    # (una entrada por época) que luego graficamos y enviamos a MLflow.
    history = {"train_loss": [], "val_loss": [], "val_metric": []}
    best_val, best_state, wait = float("inf"), None, 0

    def epoch_pass(dl, train):
        model.train() if train else model.eval()
        tot, n = 0.0, 0
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for xb, yb in dl:
                if train:
                    optimizer.zero_grad()
                out = model(xb)
                loss = criterion(out, yb)
                if train:
                    loss.backward(); optimizer.step()
                tot += loss.item() * len(xb); n += len(xb)
        return tot / n

    for epoch in range(1, epochs + 1):
        tr = epoch_pass(train_dl, True)
        va = epoch_pass(val_dl, False)
        history["train_loss"].append(tr); history["val_loss"].append(va)
        if eval_metric is not None:
            history["val_metric"].append(eval_metric(model, val_dl))
        # early stopping sobre la pérdida de validación
        if va < best_val - 1e-4:
            best_val = va
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping en la época {epoch}")
                break
        if epoch % 10 == 0:
            print(f"época {epoch:3d} | train {tr:.4f} | val {va:.4f}")
    if best_state is not None:
        model.load_state_dict(best_state)
    return history, best_state


### Caso A — Regresión (California housing)

Salida **lineal** (un valor continuo) + pérdida **MSE**. Reportamos el **RMSE** en
test y registramos todo en MLflow.


In [ ]:

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

setup_mlflow("module2-05-neural-networks", backend="dagshub")

reg = fetch_california_housing()
Xr, yr = reg.data.astype("float32"), reg.target.astype("float32")
Xr_tr, Xr_tmp, yr_tr, yr_tmp = train_test_split(Xr, yr, test_size=0.3, random_state=42)
Xr_val, Xr_te, yr_val, yr_te = train_test_split(Xr_tmp, yr_tmp, test_size=0.5, random_state=42)

# Estandarizamos features (y dejamos el target tal cual, en cientos de miles de $).
xsc = StandardScaler().fit(Xr_tr)
Xr_tr, Xr_val, Xr_te = (xsc.transform(a).astype("float32") for a in (Xr_tr, Xr_val, Xr_te))

reg_train_dl, reg_val_dl, reg_test_dl = make_loaders(
    Xr_tr, yr_tr, Xr_val, yr_val, Xr_te, yr_te, batch=64, target_2d=True)
print("Regresión — train/val/test:", len(Xr_tr), len(Xr_val), len(Xr_te),
      "| features:", Xr.shape[1])


In [ ]:

HP_REG = {"task": "regression", "lr": 1e-3, "weight_decay": 1e-4, "epochs": 100,
          "batch_size": 64, "hidden": "64,32", "dropout": 0.2,
          "optimizer": "adam", "patience": 12, "output": "linear", "loss": "MSE"}

torch.manual_seed(42)
reg_model = MLP(Xr.shape[1], hidden=(64, 32), out_dim=1, p_drop=0.2)
reg_criterion = nn.MSELoss()
reg_optimizer = torch.optim.Adam(reg_model.parameters(), lr=HP_REG["lr"],
                                 weight_decay=HP_REG["weight_decay"])

def rmse_on(model, dl):
    model.eval(); se, n = 0.0, 0
    with torch.no_grad():
        for xb, yb in dl:
            pred = model(xb)
            se += ((pred - yb) ** 2).sum().item(); n += len(xb)
    return float(np.sqrt(se / n))

with mlflow.start_run(run_name="mlp-california-regression") as run:
    mlflow.log_params(HP_REG)
    hist_reg, _ = train_loop(reg_model, reg_train_dl, reg_val_dl, reg_criterion,
                             reg_optimizer, HP_REG["epochs"], HP_REG["patience"],
                             eval_metric=lambda m, dl: {"val_rmse": rmse_on(m, dl)})
    # Métricas por época -> MLflow en UN solo request (log_batch); con un
    # servidor remoto (DagsHub) esto evita ~una llamada HTTP por época.
    from mlflow.entities import Metric
    import time as _time
    _ts = int(_time.time() * 1000)
    epoch_metrics = []
    for ep, (tl, vl) in enumerate(zip(hist_reg["train_loss"], hist_reg["val_loss"]), 1):
        epoch_metrics += [Metric("train_mse", tl, _ts, ep),
                          Metric("val_mse", vl, _ts, ep),
                          Metric("val_rmse", hist_reg["val_metric"][ep - 1]["val_rmse"], _ts, ep)]
    mlflow.MlflowClient().log_batch(run.info.run_id, metrics=epoch_metrics)
    test_rmse = rmse_on(reg_model, reg_test_dl)
    mlflow.log_metric("test_rmse", test_rmse)
    # Artefactos extra: arquitectura del modelo como texto
    mlflow.log_text(str(reg_model), "model_summary.txt")
    # Log + registro EN UN SOLO PASO: registered_model_name= es la vía
    # compatible con MLflow 2/3 y DagsHub. Registrar después con una URI
    # "runs:/<id>/model" falla en MLflow 3 (los modelos son *logged models*,
    # no artefactos del run).
    reg_name = "california-housing-mlp" if registry_available() else None
    try:
        mlflow.pytorch.log_model(reg_model, name="model",
                                 input_example=Xr_te[:5],
                                 registered_model_name=reg_name)
    except TypeError:
        mlflow.pytorch.log_model(reg_model, "model",
                                 input_example=Xr_te[:5],
                                 registered_model_name=reg_name)
    print(f"Modelo registrado: '{reg_name}'" if reg_name
          else "Registry no disponible (file store) — registro omitido.")
    print(f"TEST regresión — RMSE: {test_rmse:.4f}")


In [ ]:

# Curvas por época (pérdida + RMSE de validación) y las ENVIAMOS a MLflow como
# artefacto con mlflow.log_figure (reabrimos el mismo run con su run_id).
val_rmse_curve = [m["val_rmse"] for m in hist_reg["val_metric"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(hist_reg["train_loss"], label="train")
axes[0].plot(hist_reg["val_loss"], label="val")
axes[0].set_title("Regresión — pérdida (MSE) por época")
axes[0].set_xlabel("época"); axes[0].set_ylabel("MSE"); axes[0].legend()
axes[1].plot(val_rmse_curve, color="green")
axes[1].set_title("Regresión — RMSE de validación por época")
axes[1].set_xlabel("época"); axes[1].set_ylabel("RMSE")
fig.tight_layout()

with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_figure(fig, "plots/curvas_regresion.png")
print("Figura de curvas enviada a MLflow: plots/curvas_regresion.png")
plt.show()


### Caso B — Clasificación (breast cancer)

Salida con **1 logit + sigmoid** + pérdida **BCE** (usamos `BCEWithLogitsLoss`,
numéricamente estable, que aplica la sigmoid internamente). Reportamos **accuracy**
y **ROC-AUC** y registramos en MLflow.


In [ ]:

from sklearn.datasets import load_breast_cancer
from sklearn.metrics import roc_auc_score, accuracy_score

clf = load_breast_cancer()
Xc, yc = clf.data.astype("float32"), clf.target.astype("float32")
Xc_tr, Xc_tmp, yc_tr, yc_tmp = train_test_split(
    Xc, yc, test_size=0.3, random_state=42, stratify=yc)
Xc_val, Xc_te, yc_val, yc_te = train_test_split(
    Xc_tmp, yc_tmp, test_size=0.5, random_state=42, stratify=yc_tmp)

csc = StandardScaler().fit(Xc_tr)
Xc_tr, Xc_val, Xc_te = (csc.transform(a).astype("float32") for a in (Xc_tr, Xc_val, Xc_te))

clf_train_dl, clf_val_dl, clf_test_dl = make_loaders(
    Xc_tr, yc_tr, Xc_val, yc_val, Xc_te, yc_te, batch=32, target_2d=True)
print("Clasificación — train/val/test:", len(Xc_tr), len(Xc_val), len(Xc_te),
      "| features:", Xc.shape[1])


In [ ]:

HP_CLF = {"task": "classification", "lr": 1e-3, "weight_decay": 1e-4, "epochs": 100,
          "batch_size": 32, "hidden": "64,32", "dropout": 0.3,
          "optimizer": "adam", "patience": 12, "output": "sigmoid", "loss": "BCE"}

torch.manual_seed(42)
clf_model = MLP(Xc.shape[1], hidden=(64, 32), out_dim=1, p_drop=0.3)
clf_criterion = nn.BCEWithLogitsLoss()   # aplica sigmoid internamente
clf_optimizer = torch.optim.Adam(clf_model.parameters(), lr=HP_CLF["lr"],
                                 weight_decay=HP_CLF["weight_decay"])

def clf_eval(model, dl):
    model.eval(); ys, ps = [], []
    with torch.no_grad():
        for xb, yb in dl:
            prob = torch.sigmoid(model(xb))
            ps.append(prob.numpy()); ys.append(yb.numpy())
    y = np.vstack(ys).ravel(); p = np.vstack(ps).ravel()
    acc = accuracy_score(y, (p >= 0.5).astype(int))
    auc = roc_auc_score(y, p)
    return {"accuracy": float(acc), "roc_auc": float(auc)}

with mlflow.start_run(run_name="mlp-breast-cancer-clf") as run:
    mlflow.log_params(HP_CLF)
    hist_clf, _ = train_loop(clf_model, clf_train_dl, clf_val_dl, clf_criterion,
                             clf_optimizer, HP_CLF["epochs"], HP_CLF["patience"],
                             eval_metric=clf_eval)
    # Métricas por época -> MLflow en UN solo request (log_batch)
    from mlflow.entities import Metric
    import time as _time
    _ts = int(_time.time() * 1000)
    epoch_metrics = []
    for ep, (tl, vl) in enumerate(zip(hist_clf["train_loss"], hist_clf["val_loss"]), 1):
        m = hist_clf["val_metric"][ep - 1]
        epoch_metrics += [Metric("train_bce", tl, _ts, ep),
                          Metric("val_bce", vl, _ts, ep),
                          Metric("val_accuracy", m["accuracy"], _ts, ep),
                          Metric("val_roc_auc", m["roc_auc"], _ts, ep)]
    mlflow.MlflowClient().log_batch(run.info.run_id, metrics=epoch_metrics)
    test_metrics = clf_eval(clf_model, clf_test_dl)
    mlflow.log_metrics({"test_accuracy": test_metrics["accuracy"],
                        "test_roc_auc": test_metrics["roc_auc"]})
    # Artefactos extra: arquitectura del modelo como texto
    mlflow.log_text(str(clf_model), "model_summary.txt")
    # Log + registro en un solo paso (compatible MLflow 2/3 y DagsHub) — ver
    # el comentario del caso de regresión.
    reg_name = "breast-cancer-mlp" if registry_available() else None
    try:
        mlflow.pytorch.log_model(clf_model, name="model",
                                 input_example=Xc_te[:5],
                                 registered_model_name=reg_name)
    except TypeError:
        mlflow.pytorch.log_model(clf_model, "model",
                                 input_example=Xc_te[:5],
                                 registered_model_name=reg_name)
    print(f"Modelo registrado: '{reg_name}'" if reg_name
          else "Registry no disponible (file store) — registro omitido.")
    print(f"TEST clasificación — acc: {test_metrics['accuracy']:.4f} | "
          f"ROC-AUC: {test_metrics['roc_auc']:.4f}")


In [ ]:

# Curvas por época (pérdida BCE + accuracy/ROC-AUC) enviadas a MLflow.
val_acc_curve = [m["accuracy"] for m in hist_clf["val_metric"]]
val_auc_curve = [m["roc_auc"] for m in hist_clf["val_metric"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(hist_clf["train_loss"], label="train")
axes[0].plot(hist_clf["val_loss"], label="val")
axes[0].set_title("Clasificación — pérdida (BCE) por época")
axes[0].set_xlabel("época"); axes[0].set_ylabel("BCE"); axes[0].legend()
axes[1].plot(val_acc_curve, label="accuracy")
axes[1].plot(val_auc_curve, label="ROC-AUC")
axes[1].set_title("Clasificación — métricas de validación por época")
axes[1].set_xlabel("época"); axes[1].set_ylabel("score"); axes[1].legend()
fig.tight_layout()

with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_figure(fig, "plots/curvas_clasificacion.png")
print("Figura de curvas enviada a MLflow: plots/curvas_clasificacion.png")
plt.show()


### Diagrama de la arquitectura del MLP final

Dibujamos (en código, como el resto de diagramas del notebook) la arquitectura
del MLP final de clasificación: **30 features → Linear(30→64) → Linear(64→32) →
Linear(32→1)**, cada capa oculta con **BatchNorm + ReLU + Dropout**, y un único
**logit** de salida que la sigmoid (dentro de `BCEWithLogitsLoss`) convierte en
probabilidad. El diagrama se guarda también como **artefacto del run** en MLflow.


In [ ]:

def draw_mlp_architecture(layer_sizes, layer_labels, title, max_neurons=8):
    # Diagrama del MLP dibujado en código: círculos = neuronas, líneas = pesos.
    # Las capas anchas se truncan a max_neurons círculos con "⋮" en el medio.
    fig, ax = plt.subplots(figsize=(12, 6.5))
    xs = np.linspace(0.07, 0.93, len(layer_sizes))
    pos = []
    for n, x in zip(layer_sizes, xs):
        shown = min(n, max_neurons)
        ys = np.linspace(0.86, 0.16, shown)
        pos.append((x, ys, n > max_neurons))
    for (x0, ys0, _), (x1, ys1, _) in zip(pos[:-1], pos[1:]):
        for y0 in ys0:
            for y1 in ys1:
                ax.plot([x0, x1], [y0, y1], color="#c8c8c8", lw=0.35, zorder=1)
    for li, ((x, ys, trunc), n) in enumerate(zip(pos, layer_sizes)):
        color = ("#1f77b4" if li == 0 else
                 "#d62728" if li == len(layer_sizes) - 1 else "#2ca02c")
        mid = len(ys) // 2
        for k, y in enumerate(ys):
            if trunc and k == mid:
                ax.text(x, y, "⋮", ha="center", va="center", fontsize=18, zorder=3)
            else:
                ax.scatter([x], [y], s=420, color=color, edgecolor="white",
                           linewidth=1.5, zorder=2)
        ax.text(x, 0.93, f"{n} unidades" if n > 1 else "1 unidad",
                ha="center", va="center", fontsize=10, fontweight="bold")
        ax.text(x, 0.055, layer_labels[li], ha="center", va="top", fontsize=9)
    ax.set_title(title, fontsize=13)
    ax.set_xlim(0, 1); ax.set_ylim(-0.14, 1); ax.axis("off")
    fig.tight_layout()
    return fig

n_feat = Xc.shape[1]
arch_labels = [f"Entrada\n{n_feat} features\n(estandarizadas)",
               f"Oculta 1\nLinear({n_feat}→64)\nBatchNorm + ReLU\nDropout {HP_CLF['dropout']}",
               "Oculta 2\nLinear(64→32)\nBatchNorm + ReLU\nDropout " + str(HP_CLF['dropout']),
               "Salida\nLinear(32→1)\n1 logit → sigmoid\n(BCEWithLogitsLoss)"]
fig_arch = draw_mlp_architecture([n_feat, 64, 32, 1], arch_labels,
                                 "Arquitectura del MLP final — clasificación breast cancer")

with mlflow.start_run(run_id=run.info.run_id):
    mlflow.log_figure(fig_arch, "plots/arquitectura_mlp.png")
print("Diagrama de arquitectura enviado a MLflow: plots/arquitectura_mlp.png")
plt.show()


## 9. Resumen

- Una **neurona** calcula $z=\mathbf{w}^\top\mathbf{x}+b$ y $a=\phi(z)$; apilando
  capas con no linealidades obtenemos un **MLP** capaz de aproximar funciones
  complejas.
- Las **activaciones** y sus derivadas explican el **desvanecimiento del
  gradiente**; **ReLU**, BatchNorm, conexiones residuales e inicialización cuidada
  lo mitigan.
- **Decide la arquitectura antes de programar**: profundidad, ancho, activación, y
  sobre todo el par **salida ↔ pérdida** (lineal+MSE para regresión,
  sigmoid/softmax+BCE/cross-entropy para clasificación).
- **Backprop** = regla de la cadena capa por capa; el autograd lo hace por ti.
- **SGD** baja la superficie de pérdida; la **learning rate** es el knob más
  sensible; **momentum** y **Adam** mejoran la convergencia.
- Combina **dropout + weight decay + early stopping + batchnorm** contra el
  sobreajuste.
- Registra cada run en **MLflow** y guarda el ganador. Inspecciona en
  **http://localhost:5000**.
